In [ ]:
!pip install -q arxiv
!pip install -q --upgrade "transformers[torch]"
!pip install -q --upgrade accelerate safetensors huggingface_hub

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 81.5/81.5 kB 2.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 57.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 637.3/637.3 kB 11.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.2/4.2 MB 61.8 MB/s eta 0:00:00


In [ ]:
import os
import json
import ast
import arxiv
from typing import Dict, Optional
from dataclasses import dataclass
from collections import  defaultdict, Counter

import kagglehub
import pandas as pd
import numpy as np
from torch.utils.data import Dataset, WeightedRandomSampler, DataLoader
import torch
import torch.nn as nn
import transformers
import huggingface_hub
import accelerate
import safetensors
from transformers import (AutoTokenizer,
                          AutoModelForSequenceClassification,
                          Trainer,
                          TrainingArguments,
                          set_seed,
                          AutoModel,
                          PreTrainedConfig,
                          PreTrainedModel,
                          DataCollatorWithPadding)
from transformers.utils import ModelOutput
from transformers.modeling_outputs import SequenceClassifierOutput
from sklearn.metrics import f1_score, precision_score, recall_score
from sklearn.model_selection import train_test_split

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
print("transformers:", transformers.__version__)
print("torch:", torch.__version__)
print("accelerate:", accelerate.__version__)
print("safetensors:", safetensors.__version__)
print("huggingface_hub:", huggingface_hub.__version__)

transformers: 5.5.0
torch: 2.10.0+cpu
accelerate: 1.13.0
safetensors: 0.7.0
huggingface_hub: 1.9.2


In [ ]:
import sys
print(sys.version)

3.12.13 (main, Mar  4 2026, 09:23:07) [GCC 11.4.0]


In [ ]:
SEED = 42
set_seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

## Обработка данных

In [ ]:
path = kagglehub.dataset_download("neelshah18/arxivdataset")

print("Path to dataset files:", path)

Using Colab cache for faster access to the 'arxivdataset' dataset.
Path to dataset files: /kaggle/input/arxivdataset


In [ ]:
!cp -r /root/.cache/kagglehub/datasets/neelshah18/arxivdataset/versions/2 ./

cp: cannot stat '/root/.cache/kagglehub/datasets/neelshah18/arxivdataset/versions/2': No such file or directory


In [ ]:
with open('/content/2/arxivData.json', 'r') as f:
    data = json.load(f)

In [ ]:
ARXIV_CATEGORY_MAPPER = {
    "cs.AI": "Artificial Intelligence",
    "cs.AR": "Hardware Architecture",
    "cs.CC": "Computational Complexity",
    "cs.CE": "Computational Engineering, Finance, and Science",
    "cs.CG": "Computational Geometry",
    "cs.CL": "Computation and Language",
    "cs.CR": "Cryptography and Security",
    "cs.CV": "Computer Vision and Pattern Recognition",
    "cs.CY": "Computers and Society",
    "cs.DB": "Databases",
    "cs.DC": "Distributed, Parallel, and Cluster Computing",
    "cs.DL": "Digital Libraries",
    "cs.DM": "Discrete Mathematics",
    "cs.DS": "Data Structures and Algorithms",
    "cs.ET": "Emerging Technologies",
    "cs.FL": "Formal Languages and Automata Theory",
    "cs.GL": "General Literature",
    "cs.GR": "Graphics",
    "cs.GT": "Computer Science and Game Theory",
    "cs.HC": "Human-Computer Interaction",
    "cs.IR": "Information Retrieval",
    "cs.IT": "Information Theory",
    "cs.LG": "Machine Learning",
    "cs.LO": "Logic in Computer Science",
    "cs.MA": "Multiagent Systems",
    "cs.MM": "Multimedia",
    "cs.MS": "Mathematical Software",
    "cs.NA": "Numerical Analysis",
    "cs.NE": "Neural and Evolutionary Computing",
    "cs.NI": "Networking and Internet Architecture",
    "cs.OH": "Other Computer Science",
    "cs.OS": "Operating Systems",
    "cs.PF": "Performance",
    "cs.PL": "Programming Languages",
    "cs.RO": "Robotics",
    "cs.SC": "Symbolic Computation",
    "cs.SD": "Sound",
    "cs.SE": "Software Engineering",
    "cs.SI": "Social and Information Networks",
    "cs.SY": "Systems and Control",

    "econ.EM": "Econometrics",
    "econ.GN": "General Economics",
    "econ.TH": "Theoretical Economics",

    "eess.AS": "Audio and Speech Processing",
    "eess.IV": "Image and Video Processing",
    "eess.SP": "Signal Processing",
    "eess.SY": "Systems and Control",

    "math.AC": "Commutative Algebra",
    "math.AG": "Algebraic Geometry",
    "math.AP": "Analysis of PDEs",
    "math.AT": "Algebraic Topology",
    "math.CA": "Classical Analysis and ODEs",
    "math.CO": "Combinatorics",
    "math.CT": "Category Theory",
    "math.CV": "Complex Variables",
    "math.DG": "Differential Geometry",
    "math.DS": "Dynamical Systems",
    "math.FA": "Functional Analysis",
    "math.GM": "General Mathematics",
    "math.GN": "General Topology",
    "math.GR": "Group Theory",
    "math.GT": "Geometric Topology",
    "math.HO": "History and Overview",
    "math.IT": "Information Theory",
    "math.KT": "K-Theory and Homology",
    "math.LO": "Logic",
    "math.MG": "Metric Geometry",
    "math.MP": "Mathematical Physics",
    "math.NA": "Numerical Analysis",
    "math.NT": "Number Theory",
    "math.OA": "Operator Algebras",
    "math.OC": "Optimization and Control",
    "math.PR": "Probability",
    "math.QA": "Quantum Algebra",
    "math.RA": "Rings and Algebras",
    "math.RT": "Representation Theory",
    "math.SG": "Symplectic Geometry",
    "math.SP": "Spectral Theory",
    "math.ST": "Statistics Theory",

    "astro-ph.CO": "Cosmology and Nongalactic Astrophysics",
    "astro-ph.EP": "Earth and Planetary Astrophysics",
    "astro-ph.GA": "Astrophysics of Galaxies",
    "astro-ph.HE": "High Energy Astrophysical Phenomena",
    "astro-ph.IM": "Instrumentation and Methods for Astrophysics",
    "astro-ph.SR": "Solar and Stellar Astrophysics",

    "cond-mat.dis-nn": "Disordered Systems and Neural Networks",
    "cond-mat.mes-hall": "Mesoscale and Nanoscale Physics",
    "cond-mat.mtrl-sci": "Materials Science",
    "cond-mat.other": "Other Condensed Matter",
    "cond-mat.quant-gas": "Quantum Gases",
    "cond-mat.soft": "Soft Condensed Matter",
    "cond-mat.stat-mech": "Statistical Mechanics",
    "cond-mat.str-el": "Strongly Correlated Electrons",
    "cond-mat.supr-con": "Superconductivity",

    "gr-qc": "General Relativity and Quantum Cosmology",
    "hep-ex": "High Energy Physics - Experiment",
    "hep-lat": "High Energy Physics - Lattice",
    "hep-ph": "High Energy Physics - Phenomenology",
    "hep-th": "High Energy Physics - Theory",
    "math-ph": "Mathematical Physics",

    "nlin.AO": "Adaptation and Self-Organizing Systems",
    "nlin.CD": "Chaotic Dynamics",
    "nlin.CG": "Cellular Automata and Lattice Gases",
    "nlin.PS": "Pattern Formation and Solitons",
    "nlin.SI": "Exactly Solvable and Integrable Systems",

    "nucl-ex": "Nuclear Experiment",
    "nucl-th": "Nuclear Theory",

    "physics.acc-ph": "Accelerator Physics",
    "physics.ao-ph": "Atmospheric and Oceanic Physics",
    "physics.app-ph": "Applied Physics",
    "physics.atm-clus": "Atomic and Molecular Clusters",
    "physics.atom-ph": "Atomic Physics",
    "physics.bio-ph": "Biological Physics",
    "physics.chem-ph": "Chemical Physics",
    "physics.class-ph": "Classical Physics",
    "physics.comp-ph": "Computational Physics",
    "physics.data-an": "Data Analysis, Statistics and Probability",
    "physics.ed-ph": "Physics Education",
    "physics.flu-dyn": "Fluid Dynamics",
    "physics.gen-ph": "General Physics",
    "physics.geo-ph": "Geophysics",
    "physics.hist-ph": "History and Philosophy of Physics",
    "physics.ins-det": "Instrumentation and Detectors",
    "physics.med-ph": "Medical Physics",
    "physics.optics": "Optics",
    "physics.plasm-ph": "Plasma Physics",
    "physics.pop-ph": "Popular Physics",
    "physics.soc-ph": "Physics and Society",
    "physics.space-ph": "Space Physics",

    "quant-ph": "Quantum Physics",

    "q-bio.BM": "Biomolecules",
    "q-bio.CB": "Cell Behavior",
    "q-bio.GN": "Genomics",
    "q-bio.MN": "Molecular Networks",
    "q-bio.NC": "Neurons and Cognition",
    "q-bio.OT": "Other Quantitative Biology",
    "q-bio.PE": "Populations and Evolution",
    "q-bio.QM": "Quantitative Methods",
    "q-bio.SC": "Subcellular Processes",
    "q-bio.TO": "Tissues and Organs",

    "q-fin.CP": "Computational Finance",
    "q-fin.EC": "Economics",
    "q-fin.GN": "General Finance",
    "q-fin.MF": "Mathematical Finance",
    "q-fin.PM": "Portfolio Management",
    "q-fin.PR": "Pricing of Securities",
    "q-fin.RM": "Risk Management",
    "q-fin.ST": "Statistical Finance",
    "q-fin.TR": "Trading and Market Microstructure",

    "stat.AP": "Applications",
    "stat.CO": "Computation",
    "stat.ME": "Methodology",
    "stat.ML": "Machine Learning",
    "stat.OT": "Other Statistics",
    "stat.TH": "Statistics Theory"
}

In [ ]:
ALIAS_TO_CANONICAL = {
    "cs.NA": "math.NA",
    "cs.SY": "eess.SY",
    "math.IT": "cs.IT",
    "stat.TH": "math.ST"
}

In [ ]:
ARXIV_CATEGORIES = set(ARXIV_CATEGORY_MAPPER.keys())

In [ ]:
def remain_only_arxiv_categories(tags):
    return [tag for tag in tags if tag in ARXIV_CATEGORIES]

def canonicalize_tags(tags):
    return list({ALIAS_TO_CANONICAL.get(tag, tag) for tag in tags})

In [ ]:
data_dict = defaultdict(list)

for article in data:
    data_dict["id"].append(article["id"])
    data_dict["title"].append(article["title"])
    data_dict["abstract"].append(article["summary"])

    tags_dict = ast.literal_eval(article["tag"])
    tags = [tag["term"] for tag in tags_dict]

    tags = remain_only_arxiv_categories(tags)
    tags = canonicalize_tags(tags)

    data_dict["tags"].append(tags)

df = pd.DataFrame(data_dict)

In [ ]:
tags_counter = Counter(tag for tags in df["tags"] for tag in tags)

In [ ]:
TARGET_MIN_SAMPLES = 300
MAX_RESULTS_PER_TAG = 400
MIN_EXISTING_COUNT = 0

canonical_all_tags = sorted({ALIAS_TO_CANONICAL.get(tag, tag) for tag in ARXIV_CATEGORY_MAPPER.keys() })

need_more = {}
for tag in canonical_all_tags:
    current_count = tags_counter.get(tag, 0)
    if current_count >= MIN_EXISTING_COUNT:
        need = max(0, TARGET_MIN_SAMPLES - current_count)
    else:
        need = 0
    if need > 0:
        need_more[tag] = min(need, MAX_RESULTS_PER_TAG)

print(f"Нужно добрать по {len(need_more)} тегам")
print("Пример:", list(need_more.items())[:10])


client = arxiv.Client(
    page_size=100,
    delay_seconds=3.0,
    num_retries=5,
)

existing_ids = set(df["id"].astype(str).tolist())


def normalize_entry_id(entry_id: str) -> str:
    if entry_id is None:
        return ""
    entry_id = str(entry_id).strip()
    if "/abs/" in entry_id:
        entry_id = entry_id.split("/abs/")[-1]
    return entry_id

existing_ids_normalized = {normalize_entry_id(x) for x in existing_ids}


def fetch_articles_for_tag(tag: str, need_n: int):
    results_rows = []

    search = arxiv.Search(
        query=f"cat:{tag}",
        max_results=need_n * 3,
        sort_by=arxiv.SortCriterion.SubmittedDate,
      )

    for r in client.results(search):
        rid = normalize_entry_id(r.entry_id)

        if rid in existing_ids_normalized:
            continue

        raw_tags = list(r.categories) if r.categories is not None else []
        raw_tags = remain_only_arxiv_categories(raw_tags)
        raw_tags = canonicalize_tags(raw_tags)

        if not raw_tags:
            continue

        if tag not in raw_tags:
            continue

        results_rows.append({
            "id": rid,
            "title": r.title.strip() if r.title else "",
            "abstract": r.summary.strip().replace("\n", " ") if r.summary else "",
            "tags": raw_tags,
            })

        existing_ids_normalized.add(rid)

        if len(results_rows) >= need_n:
            break

    return results_rows


downloaded_rows = []

for i, (tag, need_n) in enumerate(sorted(need_more.items(), key=lambda x: (x[1], x[0]), reverse=True), 1):
    print(f"[{i}/{len(need_more)}] {tag}: нужно добрать {need_n}")
    rows = fetch_articles_for_tag(tag, need_n)
    downloaded_rows.extend(rows)
    print(f"  скачано {len(rows)}")

print(f"\nВсего новых строк до дедупликации на уровне DataFrame: {len(downloaded_rows)}")


if downloaded_rows:
    df_new = pd.DataFrame(downloaded_rows)
else:
    df_new = pd.DataFrame(columns=["id", "title", "abstract", "tags"])

print("Новых статей:", len(df_new))
display(df_new.head())

df_augmented = pd.concat([df, df_new], ignore_index=True)

df_augmented["id"] = df_augmented["id"].astype(str).map(normalize_entry_id)
df_augmented = df_augmented.drop_duplicates(subset="id", keep="first").reset_index(drop=True)

tags_counter_augmented = Counter(tag for tags in df_augmented["tags"] for tag in tags)

print("\nБыло / стало по нескольким тегам:")
for tag in list(sorted(need_more.keys()))[:20]:
    print(f"{tag:20s} {tags_counter.get(tag, 0):5d} -> {tags_counter_augmented.get(tag, 0):5d}")

print(f"\nРазмер исходного df: {len(df)}")
print(f"Размер дополненного df: {len(df_augmented)}")

Нужно добрать по 125 тегам
Пример: [('astro-ph.CO', 290), ('astro-ph.EP', 296), ('astro-ph.GA', 294), ('astro-ph.HE', 296), ('astro-ph.IM', 224), ('astro-ph.SR', 291), ('cond-mat.dis-nn', 174), ('cond-mat.mes-hall', 286), ('cond-mat.mtrl-sci', 288), ('cond-mat.other', 298)]
[1/125] q-fin.MF: нужно добрать 300
  скачано 300
[2/125] physics.plasm-ph: нужно добрать 300
  скачано 300
[3/125] physics.ed-ph: нужно добрать 300
  скачано 300
[4/125] physics.atom-ph: нужно добрать 300
  скачано 300
[5/125] physics.atm-clus: нужно добрать 300
  скачано 300
[6/125] physics.acc-ph: нужно добрать 300
  скачано 300
[7/125] nlin.SI: нужно добрать 300
  скачано 300
[8/125] math.SG: нужно добрать 300
  скачано 300
[9/125] math.KT: нужно добрать 300
  скачано 300
[10/125] econ.TH: нужно добрать 300
  скачано 300
[11/125] econ.GN: нужно добрать 300
  скачано 300
[12/125] q-fin.PR: нужно добрать 299
  скачано 299
[13/125] physics.pop-ph: нужно добрать 299
  скачано 299
[14/125] physics.gen-ph: нужно добра

,id,title,abstract,tags
0,2604.02035v1,Reinforcement Learning for Speculative Trading...,We study a speculative trading problem within ...,"[q-fin.MF, math.OC, cs.LG, q-fin.CP, q-fin.TR]"
1,2604.01300v1,On the mean-variance problem through the lens ...,We investigate the continuous-time Markowitz m...,"[q-fin.MF, math.OC, math.PR]"
2,2604.01299v1,Bridging classical and martingale Schrödinger ...,We investigate the martingale Schrödinger brid...,"[q-fin.MF, math.PR]"
3,2604.00178v1,Stratified adaptive sampling for derivative-fr...,There is emerging evidence that trust-region (...,"[q-fin.MF, math.OC]"
4,2603.29430v1,Ultra-short-term volatility surfaces,"Options with maturities below one week, hereaf...","[q-fin.MF, q-fin.CP]"



Было / стало по нескольким тегам:
astro-ph.CO             10 ->   606
astro-ph.EP              4 ->   468
astro-ph.GA              6 ->   606
astro-ph.HE              4 ->   548
astro-ph.IM             76 ->   714
astro-ph.SR              9 ->   714
cond-mat.dis-nn        126 ->   505
cond-mat.mes-hall       14 ->   723
cond-mat.mtrl-sci       12 ->  1010
cond-mat.other           2 ->   338
cond-mat.quant-gas       1 ->   419
cond-mat.soft            2 ->   653
cond-mat.stat-mech      84 ->  1089
cond-mat.str-el          5 ->   608
cond-mat.supr-con        4 ->   348
cs.AR                   52 ->   396
cs.CC                  196 ->   426
cs.CE                  285 ->   522
cs.CG                   94 ->   349
cs.DL                  139 ->   334

Размер исходного df:      41000
Размер дополненного df:   72526


In [ ]:
df_augmented.to_csv('arxiv_data.csv', index=False)

In [ ]:
!cp '/content/arxiv_data.csv' '/content/drive/MyDrive/ML ШАД 2/Библиотека'

## Создание датасета

In [ ]:
!cp '/content/drive/MyDrive/ML ШАД 2/Библиотека/arxiv_data.csv' '/content/arxiv_data.csv'

^C


In [ ]:
df_augmented = pd.read_csv('/content/arxiv_data.csv')
df_augmented["tags"] = df_augmented["tags"].apply(lambda x: ast.literal_eval(x) if isinstance(x, str) else x)
print(df_augmented.shape)
pd.concat([df_augmented.head(5), df_augmented.tail(5)])

(72526, 4)


,id,title,abstract,tags
0,1802.00209v1,Dual Recurrent Attention Units for Visual Ques...,We propose an architecture for VQA which utili...,"[cs.AI, cs.NE, cs.CV, cs.CL, stat.ML]"
1,1603.03827v1,Sequential Short-Text Classification with Recu...,Recent approaches based on artificial neural n...,"[cs.LG, cs.AI, cs.NE, stat.ML, cs.CL]"
2,1606.00776v2,Multiresolution Recurrent Neural Networks: An ...,We introduce the multiresolution recurrent neu...,"[cs.LG, cs.AI, cs.NE, stat.ML, cs.CL]"
3,1705.08142v2,Learning what to share between loosely related...,Multi-task learning is motivated by the observ...,"[cs.LG, cs.AI, cs.NE, stat.ML, cs.CL]"
4,1709.02349v2,A Deep Reinforcement Learning Chatbot,We present MILABOT: a deep reinforcement learn...,"[cs.LG, cs.AI, cs.NE, stat.ML, cs.CL]"
72521,2604.00699v1,Public transport in the 15-minute city,The 15-minute city is a powerful planning conc...,[physics.soc-ph]
72522,2604.00674v1,Managing the Mismatch: The Role of Flexibility...,A rapid expansion of system flexibility is ess...,"[physics.soc-ph, eess.SY]"
72523,2604.00386v1,Machine-learning extraction of size-dependent ...,Machine learning has become a useful tool for ...,[physics.soc-ph]
72524,2603.29782v1,Urban mobility enables deprivation bubble brea...,Urban deprivation is traditionally measured us...,[physics.soc-ph]
72525,2603.29722v1,"Fragmented Movements, Connected Opponents: Ana...",This study investigates the interconnectivity ...,[physics.soc-ph]


In [ ]:
def filter_rare_tags(df, min_elems: int = 300):
    tag_counter = Counter(tag for tags in df["tags"] for tag in tags)

    kept_tags = {tag for tag, cnt in tag_counter.items() if cnt >= min_elems}
    removed_tags = {tag for tag, cnt in tag_counter.items() if cnt < min_elems}

    df["tags"] = df["tags"].apply(lambda tags: [tag for tag in tags if tag in kept_tags])

    df = df[df["tags"].map(len) > 0].reset_index(drop=True)

    return df, tag_counter, kept_tags, removed_tags

In [ ]:
df_augmented, full_tag_counter, kept_tags, removed_tags = filter_rare_tags(df_augmented)
print(f"num kept fine labels: {len(kept_tags)}")
print(f"num removed fine labels: {len(removed_tags)}")
print(f"removed tags: {sorted(removed_tags)}")

num kept fine labels: 148
num removed fine labels: 3
removed tags: ['cs.GL', 'math.MP', 'q-fin.EC']


In [ ]:
all_labels = sorted(set(tag for tags in df_augmented["tags"] for tag in tags))

label2idx = {label: i for i, label in enumerate(all_labels)}
idx2label = {i: label for label, i in label2idx.items()}

In [ ]:
class ArticleDataset(Dataset):
    def __init__(self, df: pd.DataFrame, label2idx: dict,
                model_name: str = "distilbert/distilbert-base-cased",
                max_length: int = 512):
        self.articles = df.reset_index(drop=True)
        self.label2idx = label2idx
        self.labels_len = len(label2idx)
        self.max_length = max_length
        self.tokenizer = AutoTokenizer.from_pretrained(model_name)

    def __len__(self):
        return len(self.articles)

    def __getitem__(self, idx):
        article = self.articles.iloc[idx]

        tags = [self.label2idx[tag] for tag in article["tags"] if tag in self.label2idx]
        labels = torch.zeros((self.labels_len, ), dtype=torch.float32)
        labels[tags] = 1.0

        text = article['title'] + " [SEP] " + article['abstract']

        encoding = self.tokenizer(
            text,
            truncation=True,
            max_length=self.max_length,
            padding="max_length",
            return_tensors="pt")

        item = {k: v.squeeze(0) for k, v in encoding.items()}
        item["labels"] = labels
        return item


In [ ]:
train_df, valid_df = train_test_split(df_augmented, test_size=0.20, random_state=42, shuffle=True)
print(len(train_df), len(valid_df), len(all_labels))

57963 14491 148


In [ ]:
train_dataset = ArticleDataset(train_df, label2idx)
valid_dataset = ArticleDataset(valid_df, label2idx)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/465 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/49.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

## Бейзлайн модель

In [ ]:
model_name = "distilbert/distilbert-base-cased"

model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=len(all_labels),
    id2label=idx2label,
    label2id=label2idx,
    problem_type="multi_label_classification"
)

model.safetensors:   0%|          | 0.00/263M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForSequenceClassification LOAD REPORT from: distilbert/distilbert-base-cased
Key                     | Status     | 
------------------------+------------+-
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
classifier.weight       | MISSING    | 
pre_classifier.bias     | MISSING    | 
classifier.bias         | MISSING    | 
pre_classifier.weight   | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [ ]:
def compute_metrics(eval_pred):
  logits, labels = eval_pred

  probs = 1.0 / (1.0 + np.exp(-logits))
  preds = (probs >= 0.5).astype(int)

  metrics = {
      "micro_f1": f1_score(labels, preds, average="micro", zero_division=0),
      "macro_f1": f1_score(labels, preds, average="macro", zero_division=0),

      "micro_precision": precision_score(labels, preds, average="micro", zero_division=0),
      "macro_precision": precision_score(labels, preds, average="macro", zero_division=0),

      "micro_recall": recall_score(labels, preds, average="micro", zero_division=0),
      "macro_recall": recall_score(labels, preds, average="macro", zero_division=0) }

  return metrics

In [ ]:
training_args = TrainingArguments(
    output_dir="./arxiv_baseline",
    num_train_epochs=5,
    learning_rate=2e-5,
    lr_scheduler_type="cosine",
    warmup_steps=5,
    weight_decay=0.01,
    per_device_train_batch_size=16,
    log_level="error",
    logging_strategy="steps",
    logging_steps=5,
    save_strategy="epoch",
    save_total_limit=2,
    save_only_model=False,
    use_cpu=False,
    seed=42,
    eval_strategy="epoch",
    disable_tqdm=False,
    load_best_model_at_end=True,
    label_smoothing_factor=0.,
    optim="adamw_torch",
    fp16=torch.cuda.is_available()
)

In [ ]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=valid_dataset,
    compute_metrics=compute_metrics
)

In [ ]:
train_result = trainer.train()
print(train_result)

Epoch,Training Loss,Validation Loss,Micro F1,Macro F1,Micro Precision,Macro Precision,Micro Recall,Macro Recall
1,0.036334,0.031811,0.463096,0.058341,0.787280,0.142791,0.328024,0.047201
2,0.023672,0.027711,0.532522,0.199716,0.779232,0.437835,0.404466,0.150177
3,0.021241,0.026295,0.566145,0.273271,0.783856,0.613946,0.443082,0.207556
4,0.025763,0.025722,0.586162,0.325358,0.770791,0.637834,0.472889,0.252285


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

In [ ]:
!zip -r arxiv_baseline.zip /content/arxiv_baseline

  adding: content/arxiv_baseline/ (stored 0%)
  adding: content/arxiv_baseline/checkpoint-10881/ (stored 0%)
  adding: content/arxiv_baseline/checkpoint-10881/trainer_state.json (deflated 77%)
  adding: content/arxiv_baseline/checkpoint-10881/config.json (deflated 63%)
  adding: content/arxiv_baseline/checkpoint-10881/scheduler.pt (deflated 61%)
  adding: content/arxiv_baseline/checkpoint-10881/training_args.bin (deflated 53%)
  adding: content/arxiv_baseline/checkpoint-10881/rng_state.pth (deflated 26%)
  adding: content/arxiv_baseline/checkpoint-10881/optimizer.pt (deflated 16%)
  adding: content/arxiv_baseline/checkpoint-10881/scaler.pt (deflated 64%)
  adding: content/arxiv_baseline/checkpoint-10881/model.safetensors (deflated 8%)
  adding: content/arxiv_baseline/checkpoint-7254/ (stored 0%)
  adding: content/arxiv_baseline/checkpoint-7254/trainer_state.json (deflated 77%)
  adding: content/arxiv_baseline/checkpoint-7254/config.json (deflated 63%)
  adding: content/arxiv_baseline/c

In [ ]:
eval_metrics = trainer.evaluate()
print(eval_metrics)

In [ ]:
trainer.save_model("./best_arxiv_distilbert")
train_dataset.tokenizer.save_pretrained("./best_arxiv_distilbert")

In [ ]:
!cp '/content/arxiv_baseline.zip' '/content/drive/MyDrive/ML ШАД 2/Библиотека'

## Берт дообученный на научных текстах

In [ ]:
MODEL_NAME = "allenai/scibert_scivocab_cased"

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=len(all_labels),
    id2label=idx2label,
    label2id=label2idx,
    problem_type="multi_label_classification",
    use_safetensors=False
)

train_dataset = ArticleDataset(train_df, label2idx=label2idx, model_name=MODEL_NAME)
valid_dataset = ArticleDataset(valid_df, label2idx=label2idx, model_name=MODEL_NAME)

training_args = TrainingArguments(
    output_dir="./arxiv_baseline",
    num_train_epochs=3,
    learning_rate=2e-5,
    lr_scheduler_type="cosine",
    warmup_steps=5,
    weight_decay=0.01,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    log_level="error",
    logging_strategy="steps",
    logging_steps=5,
    save_strategy="epoch",
    save_total_limit=2,
    save_only_model=False,
    use_cpu=False,
    seed=42,
    eval_strategy="epoch",
    disable_tqdm=False,
    load_best_model_at_end=True,
    metric_for_best_model="macro_f1",
    greater_is_better=True,
    label_smoothing_factor=0.0,
    optim="adamw_torch",
    fp16=torch.cuda.is_available(),
    dataloader_pin_memory=torch.cuda.is_available(),
    report_to="none"
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=valid_dataset,
    compute_metrics=compute_metrics
)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: allenai/scibert_scivocab_cased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.decoder.bias               | UNEXPECTED | 
cls.predictions.decoder.weight             | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newl

In [ ]:
train_result = trainer.train()
print(train_result)

Epoch,Training Loss,Validation Loss,Micro F1,Macro F1,Micro Precision,Macro Precision,Micro Recall,Macro Recall
1,0.042495,0.039191,0.444300,0.028770,0.803595,0.038756,0.307026,0.025871
2,0.027813,0.031376,0.482607,0.058209,0.812755,0.141898,0.343197,0.047524


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

KeyboardInterrupt: 

In [ ]:
eval_metrics = trainer.evaluate()
print(eval_metrics)

In [ ]:
trainer.save_model("./best_arxiv_scibert")
train_dataset.tokenizer.save_pretrained("./best_arxiv_scibert")

In [ ]:
!zip -r arxiv_baseline.zip /content/arxiv_baseline

  adding: content/arxiv_baseline/ (stored 0%)
  adding: content/arxiv_baseline/checkpoint-10881/ (stored 0%)
  adding: content/arxiv_baseline/checkpoint-10881/trainer_state.json (deflated 77%)
  adding: content/arxiv_baseline/checkpoint-10881/config.json (deflated 63%)
  adding: content/arxiv_baseline/checkpoint-10881/scheduler.pt (deflated 61%)
  adding: content/arxiv_baseline/checkpoint-10881/training_args.bin (deflated 53%)
  adding: content/arxiv_baseline/checkpoint-10881/rng_state.pth (deflated 26%)
  adding: content/arxiv_baseline/checkpoint-10881/optimizer.pt (deflated 13%)
  adding: content/arxiv_baseline/checkpoint-10881/scaler.pt (deflated 64%)
  adding: content/arxiv_baseline/checkpoint-10881/model.safetensors (deflated 7%)
  adding: content/arxiv_baseline/checkpoint-7254/ (stored 0%)
  adding: content/arxiv_baseline/checkpoint-7254/trainer_state.json (deflated 77%)
  adding: content/arxiv_baseline/checkpoint-7254/config.json (deflated 63%)
  adding: content/arxiv_baseline/c

In [ ]:
!cp '/content/arxiv_scibert.zip' '/content/drive/MyDrive/ML ШАД 2/Библиотека'

## Берт, дообученный на научных текстах доработанный

In [ ]:
def build_sample_weights(df, label2idx, power = 0.5):
    label_counts = np.zeros(len(label2idx), dtype=np.float32)

    for tags in df["tags"]:
        uniq = set(tag for tag in tags if tag in label2idx)
        for tag in uniq:
            label_counts[label2idx[tag]] += 1.0

    label_counts = np.clip(label_counts, 1.0, None)
    inv_freq = 1.0 / np.power(label_counts, power)

    sample_weights = []
    for tags in df["tags"]:
        uniq = [tag for tag in set(tags) if tag in label2idx]
        if len(uniq) == 0:
            sample_weights.append(1.0)
        else:
            w = np.mean([inv_freq[label2idx[tag]] for tag in uniq])
            sample_weights.append(float(w))

    sample_weights = np.array(sample_weights, dtype=np.float32)
    sample_weights = sample_weights / sample_weights.mean()
    return torch.tensor(sample_weights, dtype=torch.float64)

In [ ]:
class AsymmetricLoss(nn.Module):
    def __init__(self, gamma_neg = 4.0, gamma_pos = 1.0, clip = 0.05, eps = 1e-8):
        super().__init__()
        self.gamma_neg = gamma_neg
        self.gamma_pos = gamma_pos
        self.clip = clip
        self.eps = eps

    def forward(self, logits, targets):
        targets = targets.float()

        prob = torch.sigmoid(logits)
        prob_pos = prob
        prob_neg = 1.0 - prob

        if self.clip is not None and self.clip > 0:
            shifted_prob_neg = torch.clamp(prob_neg + self.clip, max=1.0)

        loss_pos = targets * torch.log(torch.clamp(prob_pos, min=self.eps))
        loss_neg = (1.0 - targets) * torch.log(torch.clamp(shifted_prob_neg, min=self.eps))
        loss = loss_pos + loss_neg

        gamma = self.gamma_pos * targets + self.gamma_neg * (1.0 - targets)
        focal_weight = torch.pow(1.0 - (prob_pos * targets + shifted_prob_neg * (1.0 - targets)), gamma)

        loss *= focal_weight
        return -loss.mean()

In [ ]:
class SciBertMultiLabelConfig(PreTrainedConfig):
    model_type = "scibert_multilabel"

    def __init__(self, base_model_name="allenai/scibert_scivocab_cased", num_labels=0, dropout_prob=0.3, hidden_dim=768, **kwargs):
        super().__init__(**kwargs)
        self.base_model_name = base_model_name
        self.num_labels = num_labels
        self.dropout_prob = dropout_prob
        self.hidden_dim = hidden_dim


@dataclass
class MultiLabelOutput(ModelOutput):
    loss: Optional[torch.FloatTensor] = None
    logits: Optional[torch.FloatTensor] = None


class SciBertForMultiLabelClassification(PreTrainedModel):
    config_class = SciBertMultiLabelConfig

    def __init__(self, config: SciBertMultiLabelConfig):
        super().__init__(config)

        self.encoder = AutoModel.from_pretrained(config.base_model_name)
        hidden_size = self.encoder.config.hidden_size

        self.dropout = nn.Dropout(config.dropout_prob)
        self.classifier = nn.Sequential(
            nn.Linear(hidden_size, hidden_size),
            nn.GELU(),
            nn.Dropout(config.dropout_prob),
            nn.Linear(hidden_size, config.num_labels),
        )

        self.loss_fn = AsymmetricLoss(gamma_neg=4.0, gamma_pos=1.0, clip=0.05)
        self.post_init()

    def forward(
        self,
        input_ids=None,
        attention_mask=None,
        token_type_ids=None,
        labels=None,
        **kwargs
    ):
        outputs = self.encoder(
            input_ids=input_ids,
            attention_mask=attention_mask,
            token_type_ids=token_type_ids if token_type_ids is not None else None
        )

        emb = outputs.last_hidden_state[:, 0, :]
        logits = self.classifier(self.dropout(emb))

        loss = None
        if labels is not None:
            loss = self.loss_fn(logits, labels)

        return MultiLabelOutput(
            loss=loss,
            logits=logits
        )

In [ ]:
MODEL_NAME = "allenai/scibert_scivocab_cased"

train_dataset = ArticleDataset(
    train_df,
    label2idx=label2idx,
    model_name=MODEL_NAME,
    max_length=512
)

valid_dataset = ArticleDataset(
    valid_df,
    label2idx=label2idx,
    model_name=MODEL_NAME,
    max_length=512
)

data_collator = DataCollatorWithPadding(
    tokenizer=train_dataset.tokenizer,
    return_tensors="pt"
)

config = SciBertMultiLabelConfig(
    base_model_name=MODEL_NAME,
    num_labels=len(all_labels),
    dropout_prob=0.3
)

model = SciBertForMultiLabelClassification(config)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)

In [ ]:
class WeightedTrainer(Trainer):
    def get_train_dataloader(self):

        sample_weights = build_sample_weights(
            train_df,
            label2idx=label2idx,
            power=0.5
        )

        sampler = WeightedRandomSampler(
            weights=sample_weights,
            num_samples=len(sample_weights),
            replacement=True
        )

        return DataLoader(
            self.train_dataset,
            batch_size=self.args.train_batch_size,
            sampler=sampler,
            collate_fn=self.data_collator,
            num_workers=self.args.dataloader_num_workers,
            pin_memory=self.args.dataloader_pin_memory
        )

In [ ]:
def compute_metrics(eval_pred):
  logits, labels = eval_pred

  probs = 1.0 / (1.0 + np.exp(-logits))
  preds = (probs >= 0.5).astype(int)

  metrics = {
      "micro_f1": f1_score(labels, preds, average="micro", zero_division=0),
      "macro_f1": f1_score(labels, preds, average="macro", zero_division=0),

      "micro_precision": precision_score(labels, preds, average="micro", zero_division=0),
      "macro_precision": precision_score(labels, preds, average="macro", zero_division=0),

      "micro_recall": recall_score(labels, preds, average="micro", zero_division=0),
      "macro_recall": recall_score(labels, preds, average="macro", zero_division=0) }

  return metrics

In [ ]:
training_args = TrainingArguments(
    output_dir="./arxiv_ultra_scibert",
    num_train_epochs=5,
    learning_rate=2e-5,
    lr_scheduler_type="cosine",
    warmup_steps=200,
    weight_decay=0.01,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    gradient_accumulation_steps=2,
    logging_strategy="steps",
    logging_steps=50,
    save_strategy="epoch",
    eval_strategy="epoch",
    save_total_limit=2,
    save_only_model=False,
    load_best_model_at_end=True,
    metric_for_best_model="macro_f1",
    greater_is_better=True,
    fp16=torch.cuda.is_available(),
    dataloader_pin_memory=torch.cuda.is_available(),
    dataloader_num_workers=2,
    use_cpu=False,
    seed=42,
    report_to="none"
)

In [ ]:
trainer = WeightedTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=valid_dataset,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    processing_class=train_dataset.tokenizer
)

train_result = trainer.train()
print(train_result)

Epoch,Training Loss,Validation Loss,Micro F1,Macro F1,Micro Precision,Macro Precision,Micro Recall,Macro Recall
1,0.010641,0.004508,0.576332,0.485747,0.462329,0.401687,0.764958,0.659967
2,0.008401,0.004347,0.595282,0.513651,0.482228,0.418443,0.777578,0.690243
3,0.007532,0.004368,0.603406,0.523471,0.494174,0.429611,0.774630,0.689445
4,0.006514,0.004384,0.606063,0.529247,0.493437,0.429424,0.785308,0.704682
5,0.006548,0.004388,0.602301,0.527066,0.486418,0.425047,0.790666,0.707849


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=18115, training_loss=0.00919485052185364, metrics={'train_runtime': 8510.3857, 'train_samples_per_second': 34.054, 'train_steps_per_second': 2.129, 'total_flos': 7.687930031843328e+16, 'train_loss': 0.00919485052185364, 'epoch': 5.0})


In [ ]:
save_dir = "./best_arxiv_ultra_scibert"
trainer.save_model(save_dir)
train_dataset.tokenizer.save_pretrained(save_dir)

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('./best_arxiv_ultra_scibert/tokenizer_config.json',
 './best_arxiv_ultra_scibert/tokenizer.json')

In [ ]:
!zip -r best_arxiv_ultra_scibert.zip best_arxiv_ultra_scibert

  adding: best_arxiv_ultra_scibert/ (stored 0%)
  adding: best_arxiv_ultra_scibert/model.safetensors (deflated 7%)
  adding: best_arxiv_ultra_scibert/config.json (deflated 76%)
  adding: best_arxiv_ultra_scibert/tokenizer_config.json (deflated 41%)
  adding: best_arxiv_ultra_scibert/tokenizer.json (deflated 70%)
  adding: best_arxiv_ultra_scibert/training_args.bin (deflated 53%)


In [ ]:
pred = trainer.predict(valid_dataset)

logits = pred.predictions
labels = pred.label_ids

probs = 1.0 / (1.0 + np.exp(-logits))

best_thr = 0.5
best_macro_f1 = -1.0
best_metrics = None

for thr in np.arange(0.10, 0.91, 0.05):
    preds = (probs >= thr).astype(int)

    macro_f1 = f1_score(labels, preds, average="macro", zero_division=0)
    micro_f1 = f1_score(labels, preds, average="micro", zero_division=0)
    macro_precision = precision_score(labels, preds, average="macro", zero_division=0)
    macro_recall = recall_score(labels, preds, average="macro", zero_division=0)

    if macro_f1 > best_macro_f1:
        best_macro_f1 = macro_f1
        best_thr = float(thr)
        best_metrics = {
            "macro_f1": macro_f1,
            "micro_f1": micro_f1,
            "macro_precision": macro_precision,
            "macro_recall": macro_recall
        }

print("Best global threshold:", best_thr)
print("Best metrics:", best_metrics)

Best global threshold: 0.6000000000000002
Best metrics: {'macro_f1': 0.5608239469908927, 'micro_f1': 0.6585763962813144, 'macro_precision': 0.5643536797189697, 'macro_recall': 0.5696204686841394}


In [ ]:
save_dir = "./best_arxiv_ultra_scibert"

with open(os.path.join(save_dir, "id2label.json"), "w", encoding="utf-8") as f:
    json.dump({str(i): label for i, label in idx2label.items()}, f, ensure_ascii=False, indent=2)

with open(os.path.join(save_dir, "label2id.json"), "w", encoding="utf-8") as f:
    json.dump({label: int(i) for label, i in label2idx.items()}, f, ensure_ascii=False, indent=2)

## Двухуровневая классификация: высокоуровневые и низкоуровневые теги

In [ ]:
def compute_metrics(eval_pred):
    predictions = eval_pred.predictions
    label_ids = eval_pred.label_ids

    fine_logits, main_logits = predictions
    fine_labels, main_labels = label_ids

    fine_probs = 1.0 / (1.0 + np.exp(-fine_logits))
    fine_preds = (fine_probs >= 0.5).astype(int)

    main_probs = 1.0 / (1.0 + np.exp(-main_logits))
    main_preds = (main_probs >= 0.5).astype(int)

    return {
        "fine_micro_f1": f1_score(fine_labels, fine_preds, average="micro", zero_division=0),
        "fine_macro_f1": f1_score(fine_labels, fine_preds, average="macro", zero_division=0),
        "main_micro_f1": f1_score(main_labels, main_preds, average="micro", zero_division=0),
        "main_macro_f1": f1_score(main_labels, main_preds, average="macro", zero_division=0),
        "fine_micro_precision": precision_score(fine_labels, fine_preds, average="micro", zero_division=0),
        "fine_macro_precision": precision_score(fine_labels, fine_preds, average="macro", zero_division=0),
        "fine_micro_recall": recall_score(fine_labels, fine_preds, average="micro", zero_division=0),
        "fine_macro_recall": recall_score(fine_labels, fine_preds, average="macro", zero_division=0),
        "main_micro_precision": precision_score(main_labels, main_preds, average="micro", zero_division=0),
        "main_macro_precision": precision_score(main_labels, main_preds, average="macro", zero_division=0),
        "main_micro_recall": recall_score(main_labels, main_preds, average="micro", zero_division=0),
        "main_macro_recall": recall_score(main_labels, main_preds, average="macro", zero_division=0),
    }

In [ ]:
num_fine_labels = len(all_labels)

In [ ]:
MAIN_CATEGORY_PREFIXES = {
    "cs",
    "math",
    "stat",
    "physics",
    "astro-ph",
    "cond-mat",
    "q-bio",
    "q-fin",
    "econ",
    "eess",
    "nlin",
    "hep-ph",
    "hep-th",
    "hep-ex",
    "hep-lat",
    "nucl-th",
    "nucl-ex",
    "gr-qc",
    "quant-ph",
    "math-ph"
}

def extract_main_label(tag):
    if tag in MAIN_CATEGORY_PREFIXES:
        return tag

    for prefix in sorted(MAIN_CATEGORY_PREFIXES, key=len, reverse=True):
        if tag.startswith(prefix + "."):
            return prefix

all_main_labels = sorted({extract_main_label(tag) for tag in all_labels})
main_label2idx = {label: i for i, label in enumerate(all_main_labels)}
main_idx2label = {i: label for label, i in main_label2idx.items()}
num_main_labels = len(all_main_labels)

print("num_main_labels =", num_main_labels)
print("main labels:", all_main_labels)

num_main_labels = 20
main labels: ['astro-ph', 'cond-mat', 'cs', 'econ', 'eess', 'gr-qc', 'hep-ex', 'hep-lat', 'hep-ph', 'hep-th', 'math', 'math-ph', 'nlin', 'nucl-ex', 'nucl-th', 'physics', 'q-bio', 'q-fin', 'quant-ph', 'stat']


In [ ]:
class HierarchicalArticleDataset(Dataset):
    def __init__(
            self,
            df,
            label2idx,
            main_label2idx,
            model_name = "allenai/scibert_scivocab_cased",
            max_length = 512
        ):
        self.articles = df.reset_index(drop=True)
        self.label2idx = label2idx
        self.main_label2idx = main_label2idx
        self.num_fine_labels = len(label2idx)
        self.num_main_labels = len(main_label2idx)
        self.max_length = max_length
        self.tokenizer = AutoTokenizer.from_pretrained(model_name)

    def __len__(self):
        return len(self.articles)

    def __getitem__(self, idx):
        article = self.articles.iloc[idx]

        fine_tags = [tag for tag in article["tags"] if tag in self.label2idx]
        main_tags = sorted({extract_main_label(tag) for tag in fine_tags})

        fine_labels = torch.zeros(self.num_fine_labels, dtype=torch.float32)
        main_labels = torch.zeros(self.num_main_labels, dtype=torch.float32)

        fine_indices = [self.label2idx[tag] for tag in fine_tags]
        main_indices = [self.main_label2idx[tag] for tag in main_tags]

        if fine_indices:
            fine_labels[fine_indices] = 1.0
        if main_indices:
            main_labels[main_indices] = 1.0

        encoding = self.tokenizer(
            article["title"],
            article["abstract"],
            truncation=True,
            max_length=self.max_length,
            padding="max_length",
            return_tensors="pt"
        )

        item = {k: v.squeeze(0) for k, v in encoding.items()}
        item["labels"] = fine_labels
        item["main_labels"] = main_labels
        return item

In [ ]:
MODEL_NAME = "allenai/scibert_scivocab_cased"

train_dataset = HierarchicalArticleDataset(
    train_df,
    label2idx=label2idx,
    main_label2idx=main_label2idx,
    model_name=MODEL_NAME,
    max_length=512
)

valid_dataset = HierarchicalArticleDataset(
    valid_df,
    label2idx=label2idx,
    main_label2idx=main_label2idx,
    model_name=MODEL_NAME,
    max_length=512
)

In [ ]:
def build_multilabel_pos_weight(df, label2idx, tag_extractor = None, min_count = 1.0, max_pos_weight = 50.0):
    counts = np.zeros(len(label2idx), dtype=np.float32)
    n_samples = len(df)

    for tags in df["tags"]:
        if tag_extractor is None:
            current_labels = set(tag for tag in tags if tag in label2idx)
        else:
            current_labels = set(tag_extractor(tags))
            current_labels = set(tag for tag in current_labels if tag in label2idx)

        for label in current_labels:
            counts[label2idx[label]] += 1.0

    counts = np.clip(counts, min_count, None)

    pos_weight = (n_samples - counts) / counts
    pos_weight = np.clip(pos_weight, 1.0, max_pos_weight)

    return torch.tensor(pos_weight, dtype=torch.float32)


fine_pos_weight = build_multilabel_pos_weight(train_df, label2idx)

print(f"fine_pos_weight.shape = {fine_pos_weight.shape}")
print(f"fine_pos_weight min/max = {fine_pos_weight.min().item()}, {fine_pos_weight.max().item()}")

def extract_main_tags(tags):
    return {extract_main_label(tag) for tag in tags}

main_pos_weight = build_multilabel_pos_weight(train_df,main_label2idx, extract_main_tags)

print(f"main_pos_weight.shape = {main_pos_weight.shape}")
print(f"main_pos_weight min/max = {main_pos_weight.min().item()}, {main_pos_weight.max().item()}")

fine_pos_weight.shape = torch.Size([148])
fine_pos_weight min/max = 3.5554070472717285, 50.0
main_pos_weight.shape = torch.Size([20])
main_pos_weight min/max = 1.0, 50.0


In [ ]:
class HierarchicalArxivConfig(PreTrainedConfig):
    model_type = "hierarchical_arxiv"

    def __init__(
        self,
        base_model_name="allenai/scibert_scivocab_cased",
        num_fine_labels=0,
        num_main_labels=0,
        main_alpha=1.0,
        dropout_prob=0.2,
        hidden_dim=512,
        main_hidden_dim=128,
        **kwargs
    ):
        super().__init__(**kwargs)
        self.base_model_name = base_model_name
        self.num_fine_labels = num_fine_labels
        self.num_main_labels = num_main_labels
        self.main_alpha = main_alpha
        self.dropout_prob = dropout_prob
        self.hidden_dim = hidden_dim
        self.main_hidden_dim = main_hidden_dim

In [ ]:
@dataclass
class HierarchicalClassifierOutput(ModelOutput):
    loss: Optional[torch.float] = None
    logits: Optional[Tuple[torch.float, torch.float]] = None


class HierarchicalArxivClassifier(PreTrainedModel):
    config_class = HierarchicalArxivConfig

    def __init__(
        self,
        config,
        fine_pos_weight = None,
        main_pos_weight = None
    ):
        super().__init__(config)

        self.encoder = AutoModel.from_pretrained(config.base_model_name)
        hidden_size = self.encoder.config.hidden_size

        self.dropout = nn.Dropout(config.dropout_prob)

        self.layer = nn.Sequential(
            nn.Linear(hidden_size, config.hidden_dim),
            nn.GELU(),
            nn.Dropout(config.dropout_prob)
        )

        self.main_head = nn.Linear(config.hidden_dim, config.num_main_labels)

        self.main_features = nn.Sequential(
            nn.Linear(config.num_main_labels, config.main_hidden_dim),
            nn.GELU(),
            nn.Dropout(config.dropout_prob)
        )

        self.fine_head = nn.Linear(
            config.hidden_dim + config.main_hidden_dim,
            config.num_fine_labels
        )

        self.main_alpha = config.main_alpha

        self.fine_loss_fn = nn.BCEWithLogitsLoss(
            pos_weight=fine_pos_weight if fine_pos_weight is not None else None
        )
        self.main_loss_fn = nn.BCEWithLogitsLoss(
            pos_weight=main_pos_weight if main_pos_weight is not None else None
        )

        self.post_init()

    def forward(
        self,
        input_ids=None,
        attention_mask=None,
        token_type_ids=None,
        labels=None,
        main_labels=None,
        **kwargs
    ):
        outputs = self.encoder(
            input_ids=input_ids,
            attention_mask=attention_mask,
            token_type_ids=token_type_ids if token_type_ids is not None else None
        )

        emb = outputs.last_hidden_state[:, 0, :]
        z = self.layer(self.dropout(emb))

        main_logits = self.main_head(z)
        main_probs = torch.sigmoid(main_logits)
        main_features = self.main_features(main_probs)

        fine_input = torch.cat([z, main_features], dim=-1)
        fine_logits = self.fine_head(fine_input)

        loss = None
        if labels is not None:
            fine_loss = self.fine_loss_fn(fine_logits, labels)
            if main_labels is not None:
                main_loss = self.main_loss_fn(main_logits, main_labels)
                loss = fine_loss + self.main_alpha * main_loss
            else:
                loss = fine_loss

        return HierarchicalClassifierOutput(
            loss=loss,
            logits=(fine_logits, main_logits)
        )

In [ ]:
config = HierarchicalArxivConfig(
    base_model_name=MODEL_NAME,
    num_fine_labels=num_fine_labels,
    num_main_labels=num_main_labels,
    main_alpha=1.0,
    dropout_prob=0.2,
    hidden_dim=512,
    main_hidden_dim=128
)

model = HierarchicalArxivClassifier(
    config=config,
    fine_pos_weight=fine_pos_weight,
    main_pos_weight=main_pos_weight
)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: allenai/scibert_scivocab_cased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.decoder.bias               | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.decoder.weight             | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [ ]:
training_args = TrainingArguments(
    output_dir="./arxiv_hierarchical_scibert",
    label_names=["labels", "main_labels"],
    num_train_epochs=10,
    learning_rate=3e-5,
    lr_scheduler_type="cosine",
    warmup_steps=500,
    weight_decay=0.01,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    gradient_accumulation_steps=2,
    logging_strategy="steps",
    logging_steps=50,
    save_strategy="epoch",
    eval_strategy="epoch",
    save_total_limit=3,
    save_only_model=False,
    load_best_model_at_end=True,
    metric_for_best_model="fine_macro_f1",
    greater_is_better=True,
    fp16=torch.cuda.is_available(),
    dataloader_pin_memory=torch.cuda.is_available(),
    use_cpu=False,
    seed=42,
    report_to="none"
)

In [ ]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=valid_dataset,
    compute_metrics=compute_metrics,
    processing_class=train_dataset.tokenizer
)

In [ ]:
train_result = trainer.train()
print(train_result)

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'pad_token_id': 0}.


Epoch,Training Loss,Validation Loss,Fine Micro F1,Fine Macro F1,Main Micro F1,Main Macro F1,Fine Micro Precision,Fine Macro Precision,Fine Micro Recall,Fine Macro Recall,Main Micro Precision,Main Macro Precision,Main Micro Recall,Main Macro Recall
1,1.178361,0.557488,0.287633,0.227480,0.727513,0.529811,0.172749,0.139265,0.858694,0.840759,0.602370,0.404269,0.918286,0.911095
2,1.003168,0.560128,0.357354,0.287689,0.763406,0.598511,0.225439,0.182469,0.861391,0.836553,0.651116,0.475263,0.922497,0.897845
3,0.835468,0.607186,0.389045,0.315077,0.776514,0.629782,0.251704,0.203735,0.856249,0.832012,0.675020,0.512685,0.913929,0.877797


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Epoch,Training Loss,Validation Loss,Fine Micro F1,Fine Macro F1,Main Micro F1,Main Macro F1,Fine Micro Precision,Fine Macro Precision,Fine Micro Recall,Fine Macro Recall,Main Micro Precision,Main Macro Precision,Main Micro Recall,Main Macro Recall
1,1.178361,0.557488,0.287633,0.227480,0.727513,0.529811,0.172749,0.139265,0.858694,0.840759,0.602370,0.404269,0.918286,0.911095
2,1.003168,0.560128,0.357354,0.287689,0.763406,0.598511,0.225439,0.182469,0.861391,0.836553,0.651116,0.475263,0.922497,0.897845
3,0.835468,0.607186,0.389045,0.315077,0.776514,0.629782,0.251704,0.203735,0.856249,0.832012,0.675020,0.512685,0.913929,0.877797


KeyboardInterrupt: 

In [ ]:
eval_metrics = trainer.evaluate()
print(eval_metrics)

In [ ]:
save_dir = "./best_arxiv_hierarchical_scibert"
trainer.save_model(save_dir)
train_dataset.tokenizer.save_pretrained(save_dir)